In [16]:
# ============================================================
# CELL 0: Quick environment + dataset path sanity checks
# ============================================================
from pathlib import Path
import os
import sys

print(f"Python: {sys.version.split()[0]}")
print(f"CWD: {Path.cwd()}")

raw_dir = Path("data/raw/trash-icra19")
print(f"\nRAW_DIR exists: {raw_dir.exists()}")
if raw_dir.exists():
    top = sorted([p.name for p in raw_dir.iterdir()])
    print(f"RAW_DIR entries ({len(top)}): {top[:50]}")
    # one level deep
    for child in list(raw_dir.iterdir())[:10]:
        if child.is_dir():
            entries = sorted([p.name for p in child.iterdir()])
            print(f"  - {child.name}/ ({len(entries)} entries) e.g. {entries[:20]}")


Python: 3.12.13
CWD: /content/turb-detr

RAW_DIR exists: True
RAW_DIR entries (4): ['Realworld-Underwater-Image-Enhancement-RUIE-Benchmark-master', 'TrashCAN 1.0', 'UFO-120', 'trash_ICRA19']
  - Realworld-Underwater-Image-Enhancement-RUIE-Benchmark-master/ (4 entries) e.g. ['UCCS', 'UIQS', 'UIQS.rar', 'UTTS']
  - UFO-120/ (3 entries) e.g. ['TEST', '_INFO_.txt', 'train_val']
  - trash_ICRA19/ (4 entries) e.g. ['LICENSE.txt', 'README.txt', 'cfgs_and_weights', 'dataset']
  - TrashCAN 1.0/ (5 entries) e.g. ['README.txt', 'instance_version', 'material_version', 'original_data', 'scripts']


# Turb-DETR — Week 1: Baseline + Turbidity Collapse Proof

**Goal:** Prove that turbidity breaks RT-DETR. If this fails, the project dies.

**What this notebook does (in order):**
1. Sets up environment + clones your repo
2. Downloads Trash-ICRA19 from your Google Drive
3. Validates annotations and fixes issues
4. Creates stratified train/val/test split
5. Trains vanilla RT-DETR baseline (50 epochs — sanity check)
6. Evaluates on clean test set → **Track A baseline mAP**
7. Generates turbid test images (Jaffe-McGlamery, 3 levels)
8. Evaluates baseline on turbid images → **Track B collapse proof**
9. Outputs the motivation table for your paper

**Runtime:** ~4-6 hours total on T4 GPU

---
⚠️ **Before running:** Make sure you have GPU enabled.  
`Runtime → Change runtime type → T4 GPU`


In [2]:
# ============================================================
# CELL 1: Verify GPU is available
# ==========================================================import os
import torch

# Use a single variable everywhere else in the notebook
# - In Colab, you typically want GPU
# - Locally (VS Code/Windows), CPU fallback is useful for debugging
if torch.cuda.is_available():
    DEVICE = 0
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    gpu_mem = getattr(props, "total_memory", None)
    gpu_mem_gb = (gpu_mem / 1e9) if gpu_mem is not None else None
    if gpu_mem_gb is None:
        print(f"✓ GPU: {gpu_name}")
    else:
        print(f"✓ GPU: {gpu_name} ({gpu_mem_gb:.1f} GB)")
else:
    DEVICE = "cpu"
    print("⚠️  No CUDA GPU detected; running on CPU.")

print(f"✓ PyTorch: {torch.__version__}")
print(f"✓ CUDA: {torch.version.cuda}")


✓ GPU: Tesla T4 (15.6 GB)
✓ PyTorch: 2.10.0+cu128
✓ CUDA: 12.8


In [3]:
# ============================================================
# CELL 2: Install dependencies
# ============================================================
!pip install -q ultralytics==8.3.40 wandb thop scikit-image gdown

import ultralytics
print(f"✓ Ultralytics: {ultralytics.__version__}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 898.5/898.5 kB 21.5 MB/s eta 0:00:0000:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✓ Ultralytics: 8.3.40


In [4]:
# ============================================================
# CELL 3: Clone your repo and set up paths
# ============================================================
import os

# CHANGE THIS to your actual repo URL after you push the scaffold
REPO_URL = "https://github.com/csdeepak/turb-detr-underwater-detection.git"  

PROJECT_DIR = "/content/turb-detr"

if os.path.exists(PROJECT_DIR):
    print(f"Project dir already exists at {PROJECT_DIR}")
else:
    # If repo not yet pushed, just create the structure
    os.makedirs(f"{PROJECT_DIR}/data/raw", exist_ok=True)
    os.makedirs(f"{PROJECT_DIR}/data/processed", exist_ok=True)
    os.makedirs(f"{PROJECT_DIR}/data/splits", exist_ok=True)
    os.makedirs(f"{PROJECT_DIR}/data/augmented", exist_ok=True)
    os.makedirs(f"{PROJECT_DIR}/results/tables", exist_ok=True)
    os.makedirs(f"{PROJECT_DIR}/results/figures", exist_ok=True)
    os.makedirs(f"{PROJECT_DIR}/results/qualitative", exist_ok=True)
    print(f"✓ Created project structure at {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print(f"✓ Working directory: {os.getcwd()}")


✓ Created project structure at /content/turb-detr
✓ Working directory: /content/turb-detr


## Step 1 — Download Trash-ICRA19

Your dataset is on Google Drive. We download it directly.

**If gdown fails:** manually upload the dataset to `/content/turb-detr/data/raw/`


In [ ]:
# ============================================================
# CELL 4: Download Trash-ICRA19 from Google Drive (gdown)
# ============================================================
import os
from pathlib import Path
import gdown

# Your Google Drive folder ID
DRIVE_FOLDER_ID = "1pFQ7s1dKNcCs8XoAqVko91mM_j2sHVSE"
RAW_DIR = Path("data/raw/trash-icra19")

RAW_DIR.mkdir(parents=True, exist_ok=True)
existing = list(RAW_DIR.iterdir())
if existing:
    print(f"Dataset already exists at {RAW_DIR}")
    print(f"Entries: {len(existing)}")
else:
    print("Downloading from Google Drive...")
    print("(If this ends up empty, run the next cell to link from Drive manually.)")
    try:
        gdown.download_folder(
            id=DRIVE_FOLDER_ID,
            output=str(RAW_DIR),
            quiet=False,
        )
    except Exception as e:
        print(f"\n✗ gdown failed: {e}")

    # Verify something actually landed
    after = list(RAW_DIR.iterdir())
    if not after:
        print("\n⚠️  Download completed but RAW_DIR is still empty.")
        print("This usually means the Drive folder is not publicly accessible, or gdown couldn't traverse it.")
        print("Run the next cell to mount Drive and symlink/copy the dataset instead.")
    else:
        print(f"\n✓ Downloaded to {RAW_DIR} ({len(after)} entries)")


(If this fails, use the manual mount method in the next cell)


Retrieving folder contents


Retrieving folder 1HXqZNlEWX9Vnz9FUoTXOl5PNK8vRSsEM Realworld-Underwater-Image-Enhancement-RUIE-Benchmark-master
Retrieving folder 15jUjya1GthzUPozCHXh0GzJjjC9zS48Q UCCS
Retrieving folder 1k6BRkWZpDjxvH5Nv7FdITooFUlt8q8jb blue
Processing file 1eKGwJMgicXLM4SijK-dPCvFENnZgIZU_ blue_00.jpg
Processing file 1bz2rcg9p_SIKVxSpZDVlV2Rt3sUZr_Fi blue_01.jpg
Processing file 1kNR6wEZWfPUma4lC7wvaxQWBUtiGuUZF blue_02.jpg
Processing file 1iGZZ6xFQ_JAlGUNU0Bo4oB74y7xPu9-_ blue_03.jpg
Processing file 1kmn95xCTTndGQ1u8_N4jtOSRJzA5tvwp blue_04.jpg
Processing file 16fFZlUrGFP5CPNg7X7gW7dFbh1AkXSie blue_05.jpg
Processing file 10WbUfs38EIojkoZIHCJZ7Er-b2y8iady blue_06.jpg
Processing file 1Qdet4LFis-HQbd831Bbwy4drqukzWf9U blue_07.jpg
Processing file 1-TwaupuVsXutqYb5Drcrsngt9ehSmWBB blue_08.jpg
Processing file 18N_wKTRR6xhP_gHSlqpva-B0IYLJCtWF blue_9.jpg
Processing file 1hstvGCxUOnb0W2_reW55d8ly9rV3TbjZ blue_10.jpg
Processing file 1zL6MELp1XsOwpGJ2wYMfdEPsMnW90pgm blue_11.jpg
Processing file 1lg9vSTn39NXha

In [15]:
# ============================================================
# CELL 4b: FALLBACK — Mount Google Drive and link/copy dataset
# ============================================================
import os
import shutil
from pathlib import Path

# Colab-only. If you're not in Colab, skip this cell.
try:
    from google.colab import drive  # type: ignore
except Exception as e:
    raise RuntimeError("This cell requires Google Colab (google.colab).") from e

drive.mount('/content/drive', force_remount=False)

RAW_DIR = Path("data/raw/trash-icra19")

# Optional: if you know the exact dataset root folder on Drive, set it here
# Example: DRIVE_DATASET_PATH = Path("/content/drive/MyDrive/underwater_datasets/trash-icra19")
DRIVE_DATASET_PATH = None  # leave None to auto-detect

# Search roots (many people keep data in shared drives or via shortcuts)
SEARCH_ROOTS = [
    Path("/content/drive/MyDrive"),
    Path("/content/drive/Shareddrives"),
    Path("/content/drive/.shortcut-targets-by-id"),
    Path("/content/drive/MyDrive/shortcuts"),
    Path("/content/drive/MyDrive/underwater_datasets"),
    Path("/content/drive/MyDrive/datasets"),
    Path("/content/drive/MyDrive/data"),
    Path("/content/drive/MyDrive/Datasets"),
    Path("/content/drive/MyDrive/Data"),
    Path("/content/drive/MyDrive/drive"),
    Path("/content/drive/MyDrive/Drive"),
    Path("/content/drive/MyDrive/Shared"),
    Path("/content/drive/MyDrive/shared"),
    Path("/content/drive/MyDrive/Underwater"),
    Path("/content/drive/MyDrive/underwater"),
    Path("/content/drive/MyDrive/Trash"),
    Path("/content/drive/MyDrive/trash"),
    Path("/content/drive"),
 ]

IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

def _count_images_in_dir(p: Path) -> int:
    try:
        return sum(1 for f in p.iterdir() if f.is_file() and f.suffix.lower() in IMAGE_EXTS)
    except Exception:
        return 0

def _looks_like_yolo_label_file(txt_path: Path) -> bool:
    try:
        with open(txt_path, "r") as f:
            for _ in range(10):
                line = f.readline()
                if not line:
                    break
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) != 5:
                    return False
                int(parts[0])
                for v in parts[1:]:
                    float(v)
                return True
        return False
    except Exception:
        return False

def _count_label_txt_candidates_in_dir(p: Path, sample: int = 20) -> int:
    """Counts .txt files likely to be label files (YOLO or at least label-like)."""
    try:
        txts = [f for f in p.iterdir() if f.is_file() and f.suffix.lower() == ".txt"]
    except Exception:
        return 0
    if not txts:
        return 0
    # If there are many .txt files, treat it as labels even if sampling fails
    if len(txts) >= 200:
        return len(txts)
    ok = 0
    checked = 0
    for t in txts[:sample]:
        checked += 1
        if _looks_like_yolo_label_file(t):
            ok += 1
    if ok == 0:
        return 0
    return len(txts)

def _find_dataset_root_fuzzy(root: Path, max_depth: int = 10, top_k: int = 50):
    img_dirs = []  # (count, dir)
    lbl_dirs = []  # (count, dir)
    root = root.resolve()

    for dirpath, dirnames, _filenames in os.walk(root):
        # Limit recursion depth
        try:
            rel = Path(dirpath).resolve().relative_to(root)
        except Exception:
            continue
        depth = len(rel.parts)
        if depth > max_depth:
            dirnames[:] = []
            continue
        p = Path(dirpath)
        ic = _count_images_in_dir(p)
        if ic:
            img_dirs.append((ic, p))
        lc = _count_label_txt_candidates_in_dir(p)
        if lc:
            lbl_dirs.append((lc, p))

    img_dirs.sort(key=lambda t: t[0], reverse=True)
    lbl_dirs.sort(key=lambda t: t[0], reverse=True)
    img_dirs = img_dirs[:top_k]
    lbl_dirs = lbl_dirs[:top_k]

    if not img_dirs or not lbl_dirs:
        return None, img_dirs[:5], lbl_dirs[:5]

    best_root = None
    best_score = -1
    best_pair = None

    for ic, ip in img_dirs:
        for lc, lp in lbl_dirs:
            common = Path(os.path.commonpath([str(ip), str(lp)]))
            if common == root:
                continue
            try:
                depth = len(common.resolve().relative_to(root).parts)
            except Exception:
                depth = 0
            if depth < 1:
                continue
            score = (ic + lc) + depth * 2
            if score > best_score:
                best_score = score
                best_root = common
                best_pair = (ip, lp, ic, lc)

    return best_root, img_dirs[:5], lbl_dirs[:5]

# If RAW_DIR exists but is an empty dir (common after gdown created it), remove it so we can link to Drive
if RAW_DIR.exists():
    if RAW_DIR.is_symlink():
        RAW_DIR.unlink()
    elif RAW_DIR.is_dir() and not any(RAW_DIR.iterdir()):
        RAW_DIR.rmdir()

RAW_DIR.parent.mkdir(parents=True, exist_ok=True)

if DRIVE_DATASET_PATH is None:
    # Print a tiny overview so you can spot your dataset folder name
    print("Drive roots available:")
    for r in [Path("/content/drive"), Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives")]:
        if r.exists():
            print(f"- {r} (sample: {sorted([p.name for p in r.iterdir()])[:15]})")

    best = None
    img_examples = []
    lbl_examples = []
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        print(f"\nSearching under: {root}")
        best, img_examples, lbl_examples = _find_dataset_root_fuzzy(root)
        if best is not None:
            DRIVE_DATASET_PATH = best
            print(f"✓ Auto-detected dataset root at: {DRIVE_DATASET_PATH}")
            break
        else:
            print("No dataset root found here.")
            if img_examples:
                print("  Top image dirs:")
                for c, p in img_examples:
                    print(f"    - {c} images in {p}")
            if lbl_examples:
                print("  Top label dirs:")
                for c, p in lbl_examples:
                    print(f"    - {c} txt in {p}")

    if DRIVE_DATASET_PATH is None:
        print("\n✗ Could not auto-detect the dataset anywhere in the searched roots.")
        print("Set DRIVE_DATASET_PATH manually near the top of this cell.")
        raise FileNotFoundError("Dataset not found on Drive via fuzzy search")
else:
    DRIVE_DATASET_PATH = Path(DRIVE_DATASET_PATH)
    if not DRIVE_DATASET_PATH.exists():
        raise FileNotFoundError(f"DRIVE_DATASET_PATH not found: {DRIVE_DATASET_PATH}")

# Link (preferred) or copy (fallback)
if RAW_DIR.exists() and RAW_DIR.is_dir() and any(RAW_DIR.iterdir()):
    print(f"RAW_DIR already populated at {RAW_DIR}; not relinking.")
else:
    try:
        os.symlink(str(DRIVE_DATASET_PATH), str(RAW_DIR))
        print(f"✓ Symlinked {RAW_DIR} → {DRIVE_DATASET_PATH}")
    except OSError as e:
        print(f"⚠️  Symlink failed ({e}). Copying instead (slower)...")
        shutil.copytree(DRIVE_DATASET_PATH, RAW_DIR, dirs_exist_ok=True)
        print(f"✓ Copied dataset into {RAW_DIR}")

print(f"\nRAW_DIR: {RAW_DIR}")
print("Sample RAW_DIR entries:", sorted([p.name for p in RAW_DIR.iterdir()])[:30])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive roots available:
- /content/drive (sample: ['.Encrypted', '.Trash-0', '.shortcut-targets-by-id', 'MyDrive'])
- /content/drive/MyDrive (sample: ['AIML_Mini_Projects_PESU.gdoc', 'AI_Powered_Xray_Research_Guide.gdoc', 'Archana_LR.gdoc', 'Architecture .gdoc', 'Big12th.xlsx', 'Brahme and ha', 'CGPA_Improvement_Goals.gsheet', 'C_S_Deepak_PES1UG23CS907_LAB4.gdoc', 'Cancer detection .gdoc', 'Capstone Project Proposal: AI-Powered X-ray Disease Detection Mobile Application.gdoc', 'Capstone Project Proposal_ AI-Powered X-ray Disease Detection Mobile Application.gdoc', 'Capstone_Report.gdoc', 'Clustering_Lab_Report_Template.gdoc', 'Colab Notebooks', 'ColabNotebooks'])

Searching under: /content/drive/MyDrive
No dataset root found here.
  Top image dirs:
    - 4 images in /content/drive/MyDrive/shortcutscore/results/exp01
    - 1 images in /content/drive/MyDrive/Col

In [17]:
# ============================================================
# CELL 5: Explore what we downloaded
# ============================================================
import os
from pathlib import Path

RAW_DIR = Path("data/raw/trash-icra19")

print("=== Dataset Structure ===")
for root, dirs, files in os.walk(RAW_DIR):
    level = root.replace(str(RAW_DIR), "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        subindent = " " * 2 * (level + 1)
        # Show file counts by extension
        ext_counts = {}
        for f in files:
            ext = Path(f).suffix.lower()
            ext_counts[ext] = ext_counts.get(ext, 0) + 1
        for ext, count in sorted(ext_counts.items()):
            print(f"{subindent}{count} {ext} files")


=== Dataset Structure ===
trash-icra19/
  Realworld-Underwater-Image-Enhancement-RUIE-Benchmark-master/
    1 .rar files
    UCCS/
      blue/
      blue-green/
      green/
    UTTS/
      pic_E/
        Annotations/
        JPEGImages/
      pic_C/
        JPEGImages/
        Annotations/
      pic_D/
        JPEGImages/
        Annotations/
      pic_B/
        JPEGImages/
        Annotations/
      pic_A/
        JPEGImages/
        Annotations/
    UIQS/
      E/
      C/
      B/
      A/
      D/
  UFO-120/
    1 .txt files
    TEST/
      mask/
      hr/
      lrd/
    train_val/
      mask/
      hr/
      lrd/
  trash_ICRA19/
    2 .txt files
    dataset/
      test/
      val/
      videos_for_testing/
      train/
    cfgs_and_weights/
      yolo/
        weights/
        cfg/
  TrashCAN 1.0/
    1 .txt files
    instance_version/
      val/
      train/
    material_version/
      train/
      val/
    scripts/
    original_data/
      images/
      annotations/


## Step 2 — Validate Annotations

Check for broken bounding boxes, missing labels, invalid class IDs.
A single bad annotation silently degrades your mAP by several points.


In [ ]:
# ============================================================
# CELL 6: Validate and fix annotations
# ============================================================
from pathlib import Path
from collections import Counter
from typing import Optional
import os

RAW_DIR = Path("data/raw/trash-icra19")

def pick_dir_with_most_files(root: Path, dirname: str, exts: set[str]) -> Optional[Path]:
    candidates = []
    for p in RAW_DIR.rglob(dirname):
        if p.is_dir():
            count = sum(1 for f in p.iterdir() if f.is_file() and f.suffix.lower() in exts)
            if count > 0:
                candidates.append((count, p))
    if not candidates:
        return None
    candidates.sort(key=lambda t: t[0], reverse=True)
    return candidates[0][1]

img_path = pick_dir_with_most_files(RAW_DIR, "images", {".jpg", ".jpeg", ".png"})
lbl_path = pick_dir_with_most_files(RAW_DIR, "labels", {".txt"})

if img_path is None or lbl_path is None:
    raise FileNotFoundError(
        f"Could not find populated 'images' and 'labels' folders under {RAW_DIR}.\n"
        "Run Cell 4/4b again and confirm the dataset is present."
    )

print(f"Using IMAGE_DIR: {img_path}")
print(f"Using LABEL_DIR: {lbl_path}")

images = sorted(f for f in img_path.iterdir() if f.suffix.lower() in {".jpg", ".jpeg", ".png"})
print(f"Total images: {len(images)}")

# Check for missing labels
missing = 0
empty = 0
bad_coords = 0
class_counts = Counter()
total_boxes = 0

for img_file in images:
    label_file = lbl_path / f"{img_file.stem}.txt"

    if not label_file.exists():
        missing += 1
        continue

    with open(label_file) as f:
        lines = [l.strip() for l in f if l.strip()]

    if not lines:
        empty += 1
        continue

    for line in lines:
        parts = line.split()
        if len(parts) != 5:
            bad_coords += 1
            continue

        try:
            cls_id = int(parts[0])
            x, y, w, h = map(float, parts[1:])
            class_counts[cls_id] += 1
            total_boxes += 1

            # Check bounds
            if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                bad_coords += 1
        except ValueError:
            bad_coords += 1

print(f"\n=== Validation Results ===")
print(f"Images:          {len(images)}")
print(f"Missing labels:  {missing}")
print(f"Empty labels:    {empty}")
print(f"Bad coordinates: {bad_coords}")
print(f"Total boxes:     {total_boxes}")
print(f"\n=== Class Distribution ===")
for cls_id in sorted(class_counts):
    print(f"  Class {cls_id}: {class_counts[cls_id]} boxes")

# Flag ROV class if present
print(f"\n⚠️  Check if any class above is ROV (should be excluded)")


In [ ]:
# ============================================================
# CELL 7: Create stratified train/val/test split (70/15/15)
# ============================================================
import random
import shutil
from pathlib import Path
from collections import defaultdict
from typing import Optional

SEED = 42
random.seed(SEED)

RAW_DIR = Path("data/raw/trash-icra19")

def pick_dir_with_most_files(root: Path, dirname: str, exts: set[str]) -> Optional[Path]:
    candidates = []
    for p in root.rglob(dirname):
        if p.is_dir():
            count = sum(1 for f in p.iterdir() if f.is_file() and f.suffix.lower() in exts)
            if count > 0:
                candidates.append((count, p))
    if not candidates:
        return None
    candidates.sort(key=lambda t: t[0], reverse=True)
    return candidates[0][1]

IMAGE_DIR = pick_dir_with_most_files(RAW_DIR, "images", {".jpg", ".jpeg", ".png"})
LABEL_DIR = pick_dir_with_most_files(RAW_DIR, "labels", {".txt"})
OUTPUT_DIR = Path("data/processed")

if IMAGE_DIR is None or LABEL_DIR is None:
    raise FileNotFoundError(
        f"Could not find populated 'images' and 'labels' folders under {RAW_DIR}."
    )

print(f"Using IMAGE_DIR: {IMAGE_DIR}")
print(f"Using LABEL_DIR: {LABEL_DIR}")

# ROV_CLASS_ID = 7  # ← Uncomment and set if ROV class exists
ROV_CLASS_ID = None

images = sorted(f for f in IMAGE_DIR.iterdir() if f.suffix.lower() in {".jpg", ".jpeg", ".png"})

# Get primary class per image for stratification
image_primary_class = {}
for img_file in images:
    label_file = LABEL_DIR / f"{img_file.stem}.txt"
    if label_file.exists():
        with open(label_file) as f:
            classes = []
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls_id = int(parts[0])
                    if ROV_CLASS_ID is not None and cls_id == ROV_CLASS_ID:
                        continue
                    classes.append(cls_id)
            image_primary_class[img_file] = classes[0] if classes else -1
    else:
        image_primary_class[img_file] = -1

# Group by primary class
class_groups = defaultdict(list)
for img_file, cls in image_primary_class.items():
    class_groups[cls].append(img_file)

# Split each class proportionally
train, val, test = [], [], []
for cls, files in class_groups.items():
    random.shuffle(files)
    n = len(files)
    n_train = int(n * 0.70)
    n_val = int(n * 0.15)
    train.extend(files[:n_train])
    val.extend(files[n_train:n_train + n_val])
    test.extend(files[n_train + n_val:])

print(f"Split: Train={len(train)} | Val={len(val)} | Test={len(test)}")
print(f"Total: {len(train) + len(val) + len(test)}")

# Copy files to processed directory
for split_name, split_files in [("train", train), ("val", val), ("test", test)]:
    img_out = OUTPUT_DIR / split_name / "images"
    lbl_out = OUTPUT_DIR / split_name / "labels"
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for img_file in split_files:
        shutil.copy2(img_file, img_out / img_file.name)
        label_file = LABEL_DIR / f"{img_file.stem}.txt"
        if label_file.exists():
            # Filter out ROV class if needed
            if ROV_CLASS_ID is not None:
                with open(label_file) as f:
                    lines = [l for l in f if not l.strip().startswith(f"{ROV_CLASS_ID} ")]
                with open(lbl_out / label_file.name, "w") as f:
                    f.writelines(lines)
            else:
                shutil.copy2(label_file, lbl_out / label_file.name)

# Save split filenames (COMMIT THESE TO GIT)
splits_dir = Path("data/splits")
splits_dir.mkdir(parents=True, exist_ok=True)
for split_name, split_files in [("train", train), ("val", val), ("test", test)]:
    with open(splits_dir / f"{split_name}.txt", "w") as f:
        f.write("\n".join(sorted(img.stem for img in split_files)) + "\n")

print(f"\n✓ Split files saved to data/splits/")
print("✓ Processed dataset saved to data/processed/")

# DATA LEAK CHECK
train_stems = {f.stem for f in train}
val_stems = {f.stem for f in val}
test_stems = {f.stem for f in test}
assert len(train_stems & test_stems) == 0, "LEAK: train/test overlap!"
assert len(train_stems & val_stems) == 0, "LEAK: train/val overlap!"
assert len(val_stems & test_stems) == 0, "LEAK: val/test overlap!"
print("✓ No data leaks detected")


In [ ]:
# ============================================================
# CELL 8: Create Ultralytics dataset YAML config
# ============================================================
import os
import yaml
from pathlib import Path

# =============================================
# UPDATE class names based on Cell 6 output
# =============================================
CLASS_NAMES = {
    0: "plastic_bag",
    1: "bottle",
    2: "can",
    3: "carton",
    4: "fishing_net",
    5: "fragment",
    6: "tire",
}
# ← Adjust these based on your actual class IDs from Cell 6 output

dataset_config = {
    "path": str(Path("data/processed").resolve()),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "names": CLASS_NAMES,
}

config_path = "configs/datasets/trash_icra19.yaml"
os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, "w") as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print(f"✓ Dataset config saved to {config_path}")
print(f"\nContents:")
with open(config_path) as f:
    print(f.read())


## Step 3 — Train Vanilla RT-DETR Baseline

This is your **control experiment**. 50 epochs for a quick sanity check.

Expected: mAP@0.5 somewhere around 65-80% (depends on class mapping and data quality).

**Estimated time: 3-6 hours on T4**


In [ ]:
# ============================================================
# CELL 9: Set seeds for reproducibility
# ============================================================
import random, numpy as np, torch, os

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f"✓ All seeds set to {SEED}")


In [ ]:
# ============================================================
# CELL 10: Train vanilla RT-DETR baseline
# ============================================================
from ultralytics import RTDETR

model = RTDETR("rtdetr-l.pt")

results = model.train(
    data="configs/datasets/trash_icra19.yaml",
    epochs=35,           # 50 = sanity check. Use 150 for publication.
    imgsz=640,
    batch=16,
    seed=42,
    project="results",
    name="baseline_rtdetr_clean",
    device=DEVICE,
    workers=2,
    patience=15,
    verbose=True,
 )

print("\n" + "="*60)
print("BASELINE TRAINING COMPLETE")
print("="*60)


## Step 4 — Track A: Evaluate on Clean Test Set

This gives you your baseline mAP. Save this number — everything else is compared to it.


In [ ]:
# ============================================================
# CELL 11: Evaluate baseline on CLEAN test set (Track A)
# ============================================================
from ultralytics import RTDETR

BEST_WEIGHTS = "results/baseline_rtdetr_clean/weights/best.pt"
model = RTDETR(BEST_WEIGHTS)

clean_results = model.val(
    data="configs/datasets/trash_icra19.yaml",
    split="test",
    device=DEVICE,
 )

clean_map50 = clean_results.box.map50
clean_map50_95 = clean_results.box.map
clean_precision = clean_results.box.mp
clean_recall = clean_results.box.mr

print("\n" + "="*60)
print("TRACK A — CLEAN TEST SET RESULTS")
print("="*60)
print(f"  mAP@0.5:      {clean_map50:.4f}")
print(f"  mAP@0.5:0.95: {clean_map50_95:.4f}")
print(f"  Precision:     {clean_precision:.4f}")
print(f"  Recall:        {clean_recall:.4f}")
print("="*60)
print()
print("SAVE THIS NUMBER. This is your baseline.")
print(f"If mAP@0.5 < 0.50: check your class mapping and annotations.")
print(f"If mAP@0.5 > 0.65: baseline is healthy. Proceed to turbidity test.")


## Step 5 — Generate Turbid Test Images (Jaffe-McGlamery)

Apply physically-grounded turbidity simulation to the **test set only**.
Same images, same labels — only the visual quality changes.

Three severity levels:
- **Light** (c=0.05): Clear coastal water
- **Medium** (c=0.15): Moderate turbidity
- **Heavy** (c=0.30): High turbidity, harbor conditions


In [ ]:
# ============================================================
# CELL 12: Jaffe-McGlamery turbidity augmentation
# ============================================================
import cv2
import numpy as np
import shutil
from pathlib import Path
from tqdm import tqdm

def jaffe_mcglamery_turbidity(image, c=0.15, depth=3.0, seed=None):
    rng = np.random.RandomState(seed)
    img = image.astype(np.float32) / 255.0

    # Beer-Lambert attenuation
    attenuation = np.exp(-c * depth)

    # Wavelength-dependent backscatter (BGR order)
    backscatter = np.zeros_like(img)
    backscatter[:, :, 0] = 0.10 * (1 - attenuation)  # Blue
    backscatter[:, :, 1] = 0.30 * (1 - attenuation)  # Green (dominant)
    backscatter[:, :, 2] = 0.05 * (1 - attenuation)  # Red (minimal)

    # Compose
    turbid = img * attenuation + backscatter

    # Particle scatter noise
    noise_std = 0.02 * c * 10
    noise = rng.normal(0, noise_std, img.shape).astype(np.float32)
    turbid = np.clip(turbid + noise, 0, 1)

    return (turbid * 255).astype(np.uint8)


# ---- Generate turbid test sets ----
LEVELS = {"light": 0.05, "medium": 0.15, "heavy": 0.30}
TEST_IMG_DIR = Path("data/processed/test/images")
TEST_LBL_DIR = Path("data/processed/test/labels")
AUG_BASE = Path("data/augmented")

test_images = sorted(TEST_IMG_DIR.glob("*"))
print(f"Test images to augment: {len(test_images)}")

for level_name, c_val in LEVELS.items():
    out_img = AUG_BASE / level_name / "images"
    out_lbl = AUG_BASE / level_name / "labels"
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)

    print(f"\nGenerating {level_name} (c={c_val})...")
    for i, img_path in enumerate(tqdm(test_images)):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        turbid = jaffe_mcglamery_turbidity(img, c=c_val, seed=42 + i)
        cv2.imwrite(str(out_img / img_path.name), turbid)

        # Copy label (same bboxes — only image changes)
        lbl_file = TEST_LBL_DIR / f"{img_path.stem}.txt"
        if lbl_file.exists():
            shutil.copy2(lbl_file, out_lbl / lbl_file.name)

print("\n✓ All turbid test sets generated")


In [ ]:
# ============================================================
# CELL 13: Visualize turbidity effect (sanity check)
# ============================================================
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

# Pick a sample image
test_images = sorted(Path("data/processed/test/images").glob("*"))
sample = test_images[0]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Clean
img_clean = cv2.imread(str(sample))
img_clean = cv2.cvtColor(img_clean, cv2.COLOR_BGR2RGB)
axes[0].imshow(img_clean)
axes[0].set_title("Clean (Original)", fontsize=14)
axes[0].axis("off")

# Turbid levels
for i, level in enumerate(["light", "medium", "heavy"]):
    turbid_path = Path(f"data/augmented/{level}/images") / sample.name
    img = cv2.imread(str(turbid_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[i+1].imshow(img)
    c_val = {"light": 0.05, "medium": 0.15, "heavy": 0.30}[level]
    axes[i+1].set_title(f"{level.capitalize()} (c={c_val})", fontsize=14)
    axes[i+1].axis("off")

plt.suptitle("Jaffe-McGlamery Turbidity Simulation", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("results/figures/turbidity_levels_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("✓ Figure saved to results/figures/turbidity_levels_comparison.png")
print("  (This goes directly in your paper's methodology section)")


## Step 6 — Track B: Evaluate Baseline on Turbid Test Sets

**This is the critical experiment.** If mAP drops significantly, your research is valid.

Expected:
| Condition | mAP@0.5 |
|-----------|---------|
| Clean | 65-80% |
| Light (c=0.05) | ~55-70% |
| Medium (c=0.15) | ~40-55% |
| Heavy (c=0.30) | ~25-40% |

If drop < 10 points at heavy → your turbidity isn't strong enough, increase c.
If drop > 15 points → you have a publishable finding.


In [ ]:
# ============================================================
# CELL 14: Create turbid dataset YAMLs and evaluate
# ============================================================
import yaml
import os
from ultralytics import RTDETR
from pathlib import Path

BEST_WEIGHTS = "results/baseline_rtdetr_clean/weights/best.pt"

# Same class names as training config
CLASS_NAMES = {
    0: "plastic_bag",
    1: "bottle",
    2: "can",
    3: "carton",
    4: "fishing_net",
    5: "fragment",
    6: "tire",
}  # ← Must match Cell 8 exactly

LEVELS = ["light", "medium", "heavy"]
C_VALUES = {"light": 0.05, "medium": 0.15, "heavy": 0.30}
results_all = {}

for level in LEVELS:
    # Create YAML config for this turbidity level
    aug_path = Path(f"data/augmented/{level}").resolve()
    config = {
        "path": str(aug_path),
        "train": "images",  # dummy — not training
        "val": "images",
        "test": "images",
        "names": CLASS_NAMES,
    }

    yaml_path = f"configs/datasets/trash_icra19_turbid_{level}.yaml"
    os.makedirs(os.path.dirname(yaml_path), exist_ok=True)
    with open(yaml_path, "w") as f:
        yaml.dump(config, f, default_flow_style=False)

    # Evaluate
    print(f"\n{'='*50}")
    print(f"Evaluating on TURBID test set: {level} (c={C_VALUES[level]})")
    print(f"{'='*50}")

    model = RTDETR(BEST_WEIGHTS)
    res = model.val(data=yaml_path, split="test", device=DEVICE)

    results_all[level] = {
        "map50": res.box.map50,
        "map50_95": res.box.map,
        "precision": res.box.mp,
        "recall": res.box.mr,
    }

    print(f"  mAP@0.5: {res.box.map50:.4f}")
    print(f"  mAP@0.5:0.95: {res.box.map:.4f}")


In [ ]:
# ============================================================
# CELL 15: YOUR FIRST PUBLISHABLE TABLE
# ============================================================
import csv
from pathlib import Path

print("\n" + "="*70)
print("TURB-DETR — MOTIVATION TABLE (Paper Table 1)")
print("="*70)
print(f"{'Condition':<20} {'mAP@0.5':>10} {'mAP@0.5:0.95':>15} {'Precision':>12} {'Recall':>10}")
print("-"*70)

# Clean results (from Cell 11)
print(f"{'Clean':<20} {clean_map50:>10.4f} {clean_map50_95:>15.4f} {clean_precision:>12.4f} {clean_recall:>10.4f}")

# Turbid results
for level in ["light", "medium", "heavy"]:
    r = results_all[level]
    print(f"{'Turbid-'+level:<20} {r['map50']:>10.4f} {r['map50_95']:>15.4f} {r['precision']:>12.4f} {r['recall']:>10.4f}")

# Compute drops
print("\n" + "="*70)
print("ACCURACY DROP ANALYSIS")
print("="*70)

for level in ["light", "medium", "heavy"]:
    drop = clean_map50 - results_all[level]["map50"]
    drop_pct = (drop / clean_map50) * 100 if clean_map50 > 0 else 0
    print(f"  {level:>8}: mAP dropped by {drop:.4f} ({drop_pct:.1f}%)")

heavy_drop = clean_map50 - results_all["heavy"]["map50"]
heavy_drop_pct = (heavy_drop / clean_map50) * 100 if clean_map50 > 0 else 0

print("\n" + "="*70)
if heavy_drop_pct > 15:
    print(f"✓ SIGNIFICANT DROP ({heavy_drop_pct:.1f}%). Your research problem is VALID.")
    print("  Proceed to SimAM implementation.")
elif heavy_drop_pct > 10:
    print(f"⚠️ MODERATE DROP ({heavy_drop_pct:.1f}%). Publishable but emphasize per-class analysis.")
    print("  Check which classes collapse hardest.")
else:
    print(f"✗ SMALL DROP ({heavy_drop_pct:.1f}%). Your turbidity may not be strong enough.")
    print("  Try increasing c values or adding depth variation.")
print("="*70)

# Save to CSV
csv_path = "results/tables/turbidity_collapse_baseline.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["condition", "map50", "map50_95", "precision", "recall"])
    writer.writeheader()
    writer.writerow({"condition": "clean", "map50": f"{clean_map50:.4f}",
                     "map50_95": f"{clean_map50_95:.4f}",
                     "precision": f"{clean_precision:.4f}",
                     "recall": f"{clean_recall:.4f}"})
    for level in ["light", "medium", "heavy"]:
        r = results_all[level]
        writer.writerow({"condition": f"turbid_{level}",
                         "map50": f"{r['map50']:.4f}",
                         "map50_95": f"{r['map50_95']:.4f}",
                         "precision": f"{r['precision']:.4f}",
                         "recall": f"{r['recall']:.4f}"})

print(f"\n✓ Results saved to {csv_path}")


In [ ]:
# ============================================================
# CELL 16: Per-class analysis (important if overall drop is moderate)
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

# Re-evaluate to get per-class results
model = RTDETR(BEST_WEIGHTS)

# Clean per-class
clean_res = model.val(data="configs/datasets/trash_icra19.yaml", split="test", device=DEVICE)
clean_per_class = clean_res.box.ap50  # AP per class at IoU=0.5

# Heavy turbid per-class
heavy_res = model.val(data="configs/datasets/trash_icra19_turbid_heavy.yaml", split="test", device=DEVICE)
heavy_per_class = heavy_res.box.ap50

class_names = list(CLASS_NAMES.values())
n_classes = min(len(class_names), len(clean_per_class))

x = np.arange(n_classes)
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, clean_per_class[:n_classes], width, label="Clean", color="#2196F3")
bars2 = ax.bar(x + width/2, heavy_per_class[:n_classes], width, label="Heavy Turbid (c=0.30)", color="#F44336")

ax.set_ylabel("AP@0.5", fontsize=12)
ax.set_title("Per-Class Detection Performance: Clean vs Heavy Turbidity", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(class_names[:n_classes], rotation=45, ha="right")
ax.legend()
ax.set_ylim(0, 1.0)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("results/figures/per_class_clean_vs_turbid.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n✓ Per-class comparison saved to results/figures/")
print("\nClasses with biggest drops need SimAM the most.")


In [ ]:
# ============================================================
# CELL 17: Download results to your local machine
# ============================================================
import shutil

# Pack results
shutil.make_archive("turb_detr_week1_results", "zip", "results")

# Pack model weights
shutil.make_archive("turb_detr_baseline_weights", "zip",
                    "results/baseline_rtdetr_clean/weights")

# Pack split files (MUST commit to git)
shutil.make_archive("turb_detr_splits", "zip", "data/splits")

print("\n=== WEEK 1 DELIVERABLES ===")
print("Download these files:")
print("  1. turb_detr_week1_results.zip  — figures + CSV tables")
print("  2. turb_detr_baseline_weights.zip — model weights")
print("  3. turb_detr_splits.zip — train/val/test split files (commit to git!)")
print()
print("=== WHAT TO REPORT BACK ===")
print("  1. mAP@0.5 on clean test set")
print("  2. mAP@0.5 on heavy turbid test set")
print("  3. The turbidity comparison figure")
print("  4. The per-class analysis figure")
print("  5. Any issues encountered (dataset format, training instability)")


## What Happens Next

**If turbidity collapse is confirmed (>15% drop):**

Week 2 tasks (in parallel):
1. **Person A:** Implement SimAM injection into Ultralytics RT-DETR source
2. **Person B:** Calibrate Jaffe-McGlamery parameters against UFO-120
3. **Person C:** Train YOLO baseline (YOLOv9n or v10n) on same data
4. **Person D:** Generate turbidity-augmented TRAINING set
5. **Person E:** Start literature re-check on arXiv

**Do NOT start Week 2 until you have the numbers from this notebook.**

---
*Turb-DETR Project — PES University, 2026*
